In [1]:
import torch.nn as nn
import numpy as np
from pathlib import Path
import torch
from torch.utils.data import Dataset, DataLoader
import torch.nn.functional as F
import time
from tokenizers import Tokenizer
from transformers import PreTrainedTokenizerFast

In [2]:
VOCAB_SIZE  = 30000 # Number of tokens in the vocabulary
SEQ_LEN     = 8      # Context Length
D_MODEL     = 8      # Input hidden dim

# ── MLA-specific dims ─────────────────────────────────────────────────────────
N_HEADS      = 2     # Number of attention heads
D_HEAD_NOPE  = 4     # NoPE (content) dim per head
D_HEAD_ROPE  = 2     # RoPE (positional) dim per head
D_HEAD       = D_HEAD_NOPE + D_HEAD_ROPE   # Effective full head dim for attention
D_C_KV       = 4    # KV latent compression dim 
D_C_Q        = 4    # Q  latent compression dim 

N_MTP      = 3    # Predict N_MTP  future tokens
MTP_LAMBDA = 0.3  # Auxiliary loss weight: Loss = Loss(next_token) + MTP_LAMBDA(Loss(mtp))

D_FF        = 16 # Expansion dim of the feedforward network in the transformer block
DROPOUT     = 0.1 # Randomly zeroes DROPOUT*100% activations during training.
BATCH_SIZE  = 64 # Each training step processes BATCH_SIZE sequences in parallel
LR          = 3e-4 # Optimizer step size
EPOCHS      = 1 # Number of times to loop through the entire training dataset

TOKENIZER_PATH = Path.cwd() / "models" / "tokenizer.json"
tokenizer = Tokenizer.from_file(str(TOKENIZER_PATH))

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"device: {DEVICE}")
print(f"input shape per sequence: ({SEQ_LEN}, {D_MODEL})")
print(f"W_DKV shape: ({D_MODEL}, {D_C_KV})")

device: cuda
input shape per sequence: (8, 8)
W_DKV shape: (8, 4)


In [3]:
# Decoupled RoPE utilities
# Applied ONLY to the Q_rope / K_rope pathways, not to the NoPE content.

def precompute_freqs_cis(d_rope: int, max_seq_len: int, theta: float = 10000.0):
    """
    Returns complex unit vectors for RoPE rotation.
    d_rope must be even.
    Output shape: (max_seq_len, d_rope // 2)  — complex64
    """
    assert d_rope % 2 == 0
    # Inverse frequencies: θ_i = 1 / 10000^(2i / d_rope)
    inv_freq = 1.0 / (theta ** (torch.arange(0, d_rope, 2).float() / d_rope))
    t        = torch.arange(max_seq_len, dtype=torch.float32) # Tells position of the token in the sequence
    # Outer product -> Angles matrix  (max_seq_len, d_rope//2),  
    angles   = torch.outer(t, inv_freq) # Value at row m and column i is the exact rotation angle in radians needed for that specific token position and feature pair.
    # e^{i·angle} = cos + i·sin  — the rotation in complex form
    freqs_cis = torch.polar(torch.ones_like(angles), angles) 
    return freqs_cis   # (T, d_rope//2)  complex64


def apply_rope(x: torch.Tensor, freqs_cis: torch.Tensor) -> torch.Tensor:
    """
    Rotate x using pre-computed complex frequencies.
    x         : (B, n_heads, T, d_rope)   — real
    freqs_cis : (T, d_rope // 2)          — complex
    Returns   : (B, n_heads, T, d_rope)   — real, rotated
    """
    B, H, T, d = x.shape
    freqs_cis = freqs_cis.to(x.device)
    # pair up consecutive dims as Re/Im of a complex number
    x_c = torch.view_as_complex(x.float().reshape(B, H, T, d // 2, 2))  # (B,H,T,d/2) complex, (xo,x1) -> (xo + i·x1)
    # broadcast freqs over batch and heads
    f   = freqs_cis[:T].unsqueeze(0).unsqueeze(0)    # (1, 1, T, d/2) complex
    # multiply = rotate in 2D plane
    out = torch.view_as_real(x_c * f).reshape(B, H, T, d)
    return out.to(x.dtype)

# Pre-compute once for the full context length
freqs_cis = precompute_freqs_cis(D_HEAD_ROPE, SEQ_LEN).to(DEVICE)
print(f"freqs_cis shape: {freqs_cis.shape}  (T={SEQ_LEN}, d_rope/2={D_HEAD_ROPE//2})  dtype={freqs_cis.dtype}")

freqs_cis shape: torch.Size([8, 1])  (T=8, d_rope/2=1)  dtype=torch.complex64


In [4]:
class MultiHeadLatentAttention(nn.Module):
    """
    DeepSeek-V2 style MLA with decoupled RoPE.
    """
    def __init__(self, d_model: int = D_MODEL, n_heads: int = N_HEADS, d_head_nope: int = D_HEAD_NOPE, d_head_rope: int = D_HEAD_ROPE,
                       d_c_kv: int = D_C_KV, d_c_q: int = D_C_Q, dropout: float = DROPOUT):
        super().__init__()
        self.n_heads     = n_heads
        self.d_head_nope = d_head_nope
        self.d_head_rope = d_head_rope
        self.d_head      = d_head_nope + d_head_rope
        self.scale       = self.d_head ** -0.5

        # ── KV pathway ───────────────────────────────────────────────────────
        self.W_DKV = nn.Linear(d_model, d_c_kv, bias=False)
        self.W_UK  = nn.Linear(d_c_kv,  n_heads*d_head_nope, bias=False)
        self.W_UV  = nn.Linear(d_c_kv,  n_heads*d_head_nope, bias=False)

        # ── Q pathway ────────────────────────────────────────────────────────
        self.W_DQ  = nn.Linear(d_model, d_c_q, bias=False)
        self.W_UQ  = nn.Linear(d_c_q,   n_heads*d_head_nope, bias=False)

        # ── Decoupled RoPE pathway ────────────────────────────────────────────
        self.W_QR = nn.Linear(d_c_q, n_heads*d_head_rope, bias=False)
        self.W_KR  = nn.Linear(d_model, d_head_rope, bias=False)  # HEAD-SHARED

        # ── Output ───────────────────────────────────────────────────────────
        self.W_O   = nn.Linear(n_heads*d_head_nope, d_model, bias=False)
        self.drop  = nn.Dropout(dropout)

    def forward(self, h: torch.Tensor, freqs_cis: torch.Tensor) -> torch.Tensor:
        """
        h         : (B, T, d_model)
        freqs_cis : (T, d_head_rope//2)  complex
        returns   : (B, T, d_model)
        """
        B, T, _ = h.shape

        # ── Step 1 · Compress ────────────────────────────────────────────────
        c_kv = self.W_DKV(h)   # (B, T, d_c_kv=4)
        c_q  = self.W_DQ(h)    # (B, T, d_c_q=4)

        # ── Step 2 · NoPE up-projection (per-head) ───────────────────────────
        def to_heads(x, d_h):
            return x.view(B, T, self.n_heads, d_h).transpose(1, 2)

        K_nope = to_heads(self.W_UK(c_kv), self.d_head_nope)  # (B, H, T, 4)
        V      = to_heads(self.W_UV(c_kv), self.d_head_nope)  # (B, H, T, 4)
        Q_nope = to_heads(self.W_UQ(c_q),  self.d_head_nope)  # (B, H, T, 4)

        # ── Step 3 · Decoupled RoPE pathway ──────────────────────────────────
        # Q_rope: per-head
        Q_rope = to_heads(self.W_QR(c_q), self.d_head_rope)
        Q_rope = apply_rope(Q_rope, freqs_cis)

        # K_rope: HEAD-SHARED — one vector per token, same for all heads
        K_rope = self.W_KR(h)                                    # (B, T, d_head_rope=2)
        K_rope = K_rope.unsqueeze(1)                             # (B, 1, T, 2)
        K_rope = apply_rope(K_rope, freqs_cis)                   # (B, 1, T, 2)  rotated
        K_rope = K_rope.expand(-1, self.n_heads, -1, -1)        # (B, H, T, 2)  broadcast

        # ── Step 4 · Concatenate NoPE + RoPE ─────────────────────────────────
        Q = torch.cat([Q_nope, Q_rope], dim=-1)  # (B, H, T, 6)
        K = torch.cat([K_nope, K_rope], dim=-1)  # (B, H, T, 6)

        # ── Step 5 · Causal attention ─────────────────────────────────────────
        attn = (Q @ K.transpose(-2, -1)) * self.scale
        mask = torch.tril(torch.ones(T, T, device=h.device, dtype=torch.bool))
        attn = attn.masked_fill(~mask, float("-inf"))
        attn = F.softmax(attn, dim=-1)
        attn = self.drop(attn)

        out = attn @ V                                            # (B, H, T, 4)
        out = out.transpose(1, 2).contiguous()
        out = out.view(B, T, self.n_heads * self.d_head_nope)
        return self.W_O(out)                                      # (B, T, d_model)

    def kv_cache_size_bytes(self, seq_len: int) -> dict:
        fp16 = 2
        # MLA caches: c_kv (shared) + K_rope (head-shared, NOT per-head)
        mla_total = (seq_len * D_C_KV + seq_len * self.d_head_rope) * fp16
        mha_total = seq_len * self.n_heads * self.d_head * fp16 * 2  # K + V
        return {
            "MHA  K+V per token": f"{self.n_heads}×{self.d_head} + {self.n_heads}×{self.d_head} = {2*self.n_heads*self.d_head} values",
            "MLA  c_kv + K_rope": f"{D_C_KV} + {self.d_head_rope} = {D_C_KV + self.d_head_rope} values  (K_rope shared, not ×{self.n_heads})",
            "bytes MHA": mha_total,
            "bytes MLA": mla_total,
            "compression_ratio": round(mha_total / mla_total, 2),
        }

In [5]:
class Expert(nn.Module):
    """A standard feed-forward expert network."""
    def __init__(self, d_model: int, d_ff: int):
        super().__init__()
        self.w1 = nn.Linear(d_model, d_ff, bias=False)
        self.w2 = nn.Linear(d_ff, d_model, bias=False)
        self.w3 = nn.Linear(d_model, d_ff, bias=False) 

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.w2(F.silu(self.w1(x)) * self.w3(x))

In [6]:
class ShallowSeekMoE(nn.Module):
    def __init__(self, 
                 d_model: int = D_MODEL, 
                 d_ff_shared: int = 16,       # Shared experts stay relatively large
                 d_ff_routed: int = 2,        # Routed experts are drastically sliced down!
                 n_shared_experts: int = 2,
                 n_routed_experts: int = 32,  # Many more experts, but each is tiny
                 top_k: int = 4,              # Route to top-k experts to recombine concepts
                 bias_update_rate: float = 0.01):
        super().__init__()
        self.d_model = d_model
        self.n_shared_experts = n_shared_experts
        self.n_routed_experts = n_routed_experts
        self.top_k = top_k
        self.bias_update_rate = bias_update_rate

        # 1. Shared Experts (Large capacity, general knowledge)
        self.shared_experts = nn.ModuleList([Expert(d_model, d_ff_shared) for _ in range(n_shared_experts)])

        # 2. Routed Experts (Tiny capacity, hyper-specialized)
        self.routed_experts = nn.ModuleList([Expert(d_model, d_ff_routed) for _ in range(n_routed_experts)])

        # 3. Router Gate Projection
        self.gate = nn.Linear(d_model, n_routed_experts, bias=False)

        # 4. Aux-Loss Free Balancing Biases
        self.register_buffer("expert_biases", torch.zeros(n_routed_experts))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        B, T, C = x.shape
        tokens_flat = x.view(-1, C) 
        num_tokens = tokens_flat.size(0)

        # ─── PATHWAY 1: SHARED EXPERTS ────────────────────────────────────────
        shared_output = torch.zeros_like(tokens_flat)
        for expert in self.shared_experts:
            shared_output += expert(tokens_flat)

        # ─── PATHWAY 2: ROUTED EXPERTS (Fine-Grained) ─────────────────────────
        logits = self.gate(tokens_flat)
        balanced_logits = logits + self.expert_biases.unsqueeze(0)

        scores = F.softmax(balanced_logits, dim=-1)
        topk_weights, topk_indices = torch.topk(scores, self.top_k, dim=-1)
        topk_weights = topk_weights / topk_weights.sum(dim=-1, keepdim=True)

        routed_output = torch.zeros_like(tokens_flat)
        actual_counts = torch.zeros(self.n_routed_experts, device=x.device)

        for i, expert in enumerate(self.routed_experts):
            mask = (topk_indices == i)
            if mask.any():
                token_indices, k_indices = torch.where(mask)
                weights = topk_weights[token_indices, k_indices].unsqueeze(-1)
                
                # Executing the tiny d_ff_routed expert
                expert_out = expert(tokens_flat[token_indices])
                routed_output[token_indices] += weights * expert_out
                
                actual_counts[i] += token_indices.size(0)

        # ─── STEP 3: BIAS UPDATE ──────────────────────────────────────────────
        if self.training:
            target_count = (num_tokens * self.top_k) / self.n_routed_experts
            error = target_count - actual_counts
            self.expert_biases += self.bias_update_rate * error

        final_output = shared_output + routed_output
        return final_output.view(B, T, C)

In [7]:
class RMSNorm(nn.Module):
    def __init__(self, d_model: int, eps: float = 1e-8):
        super().__init__()
        self.eps    = eps
        self.weight = nn.Parameter(torch.ones(d_model))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        rms = torch.rsqrt(x.pow(2).mean(-1, keepdim=True) + self.eps)
        return x * rms * self.weight

In [8]:
class TransformerBlock(nn.Module):
    """MLA attention + MoE feed-forward, both with pre-RMSNorm + residual."""
    def __init__(self,
                 d_model     = D_MODEL,
                 n_heads     = N_HEADS,
                 d_head_nope = D_HEAD_NOPE,
                 d_head_rope = D_HEAD_ROPE,
                 d_c_kv      = D_C_KV,
                 d_c_q       = D_C_Q,
                 dropout     = DROPOUT):
        super().__init__()
        self.norm1 = RMSNorm(d_model)
        self.attn  = MultiHeadLatentAttention(d_model=d_model, n_heads=n_heads, d_head_nope=d_head_nope, 
                                              d_head_rope=d_head_rope, d_c_kv=d_c_kv, d_c_q=d_c_q, dropout=dropout)
        self.norm2 = RMSNorm(d_model)
        self.moe   = ShallowSeekMoE(d_model=d_model)

    def forward(self, x: torch.Tensor, freqs_cis: torch.Tensor) -> torch.Tensor:
        x = x + self.attn(self.norm1(x), freqs_cis)   # MLA
        x = x + self.moe(self.norm2(x))               # MoE
        return x

In [9]:
class MTPModule(nn.Module):
    """
    One MTP depth-k head.

    Fuses h_prev (from the previous head / main model) with the teacher-forced
    embedding of the k-th future token, then runs a fresh TransformerBlock.
    Embedding table and LM-head weights are SHARED with the main model (tied).
    """
    def __init__(self, d_model: int, shared_embed: nn.Embedding, shared_lm_head: nn.Linear):
        super().__init__()
        self.norm_h   = RMSNorm(d_model)
        self.norm_e   = RMSNorm(d_model)
        self.proj     = nn.Linear(2 * d_model, d_model, bias=False)   # 2D → D
        self.block    = TransformerBlock(d_model=d_model)
        self.norm_out = RMSNorm(d_model)
        # references only — not re-registered as parameters
        self.shared_embed   = shared_embed
        self.shared_lm_head = shared_lm_head

    def forward(self,
                h_prev:      torch.Tensor,   # (B, T_eff, D)  — from prev head
                token_ids_k: torch.Tensor,   # (B, T_eff)     — teacher-forced offset k+1
                freqs_cis:   torch.Tensor,
               ) -> tuple[torch.Tensor, torch.Tensor]:

        e_k    = self.shared_embed(token_ids_k)                              # (B, T_eff, D)
        fused  = torch.cat([self.norm_h(h_prev), self.norm_e(e_k)], dim=-1) # (B, T_eff, 2D)
        h_in   = self.proj(fused)                                             # (B, T_eff, D)
        h_out  = self.block(h_in, freqs_cis)                                 # (B, T_eff, D)
        logits = self.shared_lm_head(self.norm_out(h_out))                   # (B, T_eff, V)
        return h_out, logits

In [10]:
class ShallowSeek(nn.Module):
    """
    embed → TransformerBlock (MLA + MoE) → LM head        ← main model
    └── N_MTP sequential MTP heads                         ← auxiliary training heads
    """
    def __init__(self, vocab_size=VOCAB_SIZE, d_model=D_MODEL, n_mtp=N_MTP):
        super().__init__()
        self.n_mtp = n_mtp

        self.embed    = nn.Embedding(vocab_size, d_model)
        self.block    = TransformerBlock(d_model=d_model)
        self.norm_out = RMSNorm(d_model)
        self.lm_head  = nn.Linear(d_model, vocab_size, bias=False)
        self.lm_head.weight = self.embed.weight              # weight tying

        self.mtp_modules = nn.ModuleList([MTPModule(d_model, self.embed, self.lm_head) for _ in range(n_mtp)])

    def forward(self, x: torch.Tensor, targets: torch.Tensor = None):
        """
        x       : (B, T)
        targets : (B, T)  — x shifted left by 1  (from dataset)
        """
        B, T = x.shape

        h           = self.embed(x)                           # (B, T, D)
        h           = self.block(h, freqs_cis)                # (B, T, D)  apply_rope slices internally
        main_logits = self.lm_head(self.norm_out(h))          # (B, T, V)

        if targets is None:
            return main_logits, None

        # Shrink to positions that have room for all n_mtp future targets
        # k=n_mtp-1 needs target at t + n_mtp + 1  →  max index = T_eff + n_mtp + 1 - 1 = T
        T_eff = T - self.n_mtp - 1   # e.g. 8 - 3 - 1 = 4

        main_loss = F.cross_entropy(main_logits[:, :T_eff].reshape(-1, main_logits.size(-1)), targets[:, :T_eff].reshape(-1))

        # ── MTP chain ─────────────────────────────────────────────────────────
        # k=0: token_ids_k = x[:,1:T_eff+1],  target = x[:,2:T_eff+2]
        # k=1: token_ids_k = x[:,2:T_eff+2],  target = x[:,3:T_eff+3]
        # k=2: token_ids_k = x[:,3:T_eff+3],  target = x[:,4:T_eff+4]  (= x[:,4:8] ✓)
        h_prev     = h[:, :T_eff]                            # (B, T_eff, D)
        mtp_losses = []

        for k, module in enumerate(self.mtp_modules):
            token_ids_k = x[:,   k + 1 : T_eff + k + 1]    # (B, T_eff)  embed input
            target_k    = x[:,   k + 2 : T_eff + k + 2]    # (B, T_eff)  prediction target

            h_prev, logits_k = module(h_prev, token_ids_k, freqs_cis)

            mtp_losses.append(F.cross_entropy(
                logits_k.reshape(-1, logits_k.size(-1)),
                target_k.reshape(-1)
            ))

        mtp_loss   = sum(mtp_losses) / self.n_mtp
        total_loss = main_loss + MTP_LAMBDA * mtp_loss
        return main_logits, total_loss

In [11]:
def load_checkpoint(path: str) -> ShallowSeek:
    ckpt = torch.load(path, map_location=DEVICE)
    vocab_size = ckpt.get("vocab_size", VOCAB_SIZE)
    model = ShallowSeek(vocab_size=vocab_size).to(DEVICE)
    model.load_state_dict(ckpt["model_state"])
    print(f"loaded epoch {ckpt['epoch']}  |  loss {ckpt['loss']:.4f}  |  vocab {vocab_size}")
    return model